# Healthcare Patient Waiting Time Analytics Pipeline

Modern Data Engineering for AI Systems - Final Project

Pipeline: Data Source → Ingestion → Spark → Data Quality → Quality Gate → Delta Lake → Analytics

## 1. Project Overview

Problem: Healthcare facilities need to understand patient waiting times and improve operational efficiency.

Goal: Build a data pipeline that cleans healthcare data, validates quality, stores trusted data, and produces analytics insights.

In [ ]:
!pip install pyspark delta-spark -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName('HealthcarePipeline') \
    .config('spark.sql.extensions','io.delta.sql.DeltaSparkSessionExtension') \
    .config('spark.sql.catalog.spark_catalog','org.apache.spark.sql.delta.catalog.DeltaCatalog') \
    .getOrCreate()

spark

## 2. Data Source
Create a sample healthcare dataset containing valid and invalid records to demonstrate Quality Gate.

In [ ]:
data = [
 (1001,45,'Emergency','High',40),
 (1002,30,'Dental','Low',20),
 (1003,60,'Cardiology','High',55),
 (1003,60,'Cardiology','High',55),
 (None,-5,'Emergency','High',70)
]

columns=['patient_id','age','department','priority','waiting_time']

df=spark.createDataFrame(data,columns)
df.show()

## 3. Data Transformation using Spark
- Remove duplicates
- Remove missing values
- Prepare trusted data

In [ ]:
clean_df=df.dropDuplicates().dropna()
clean_df.show()

## 4. Data Quality Checks
Checks implemented:
1. Completeness
2. Uniqueness
3. Validity

In [ ]:
missing_ids=df.filter(col('patient_id').isNull()).count()
duplicates=df.count()-df.dropDuplicates(['patient_id']).count()
invalid_age=df.filter((col('age')<0)|(col('age')>120)).count()

print('Missing IDs:',missing_ids)
print('Duplicates:',duplicates)
print('Invalid ages:',invalid_age)

## 5. Quality Gate
PASS → Delta Lake
FAIL → Quarantine

In [ ]:
if missing_ids==0 and duplicates==0 and invalid_age==0:
    print('PASS: Data moved to Delta Lake')
else:
    print('FAIL: Data moved to Quarantine')

## 6. Delta Lake Storage
Store validated clean healthcare data.

In [ ]:
clean_df.write.format('delta').mode('overwrite').save('/content/healthcare_delta')
print('Saved successfully')

## 7. Analytics Output
Average waiting time by department.

In [ ]:
clean_df.groupBy('department').agg(avg('waiting_time').alias('average_waiting_time')).show()

## Conclusion
The pipeline successfully demonstrates ingestion, Spark processing, quality validation, Delta Lake storage, and healthcare analytics.